In [14]:
from datetime import datetime

batch_id = datetime.utcnow().strftime("%Y%m%d_%H%M%S")
print("batch_id = ",batch_id)


StatementMeta(, 198cb09d-16bf-47c1-89e6-f5d62861830b, 16, Finished, Available, Finished, False)

batch_id =  20260312_170110


In [15]:
#处理 business 表
raw_business_path = "Files/yelp/raw/yelp_business.json"

df_business_raw = spark.read.json(raw_business_path)
df_business_raw.printSchema()
df_business_raw.count()



StatementMeta(, 198cb09d-16bf-47c1-89e6-f5d62861830b, 17, Finished, Available, Finished, False)

root
 |-- address: string (nullable = true)
 |-- attributes: struct (nullable = true)
 |    |-- AcceptsInsurance: string (nullable = true)
 |    |-- AgesAllowed: string (nullable = true)
 |    |-- Alcohol: string (nullable = true)
 |    |-- Ambience: string (nullable = true)
 |    |-- BYOB: string (nullable = true)
 |    |-- BYOBCorkage: string (nullable = true)
 |    |-- BestNights: string (nullable = true)
 |    |-- BikeParking: string (nullable = true)
 |    |-- BusinessAcceptsBitcoin: string (nullable = true)
 |    |-- BusinessAcceptsCreditCards: string (nullable = true)
 |    |-- BusinessParking: string (nullable = true)
 |    |-- ByAppointmentOnly: string (nullable = true)
 |    |-- Caters: string (nullable = true)
 |    |-- CoatCheck: string (nullable = true)
 |    |-- Corkage: string (nullable = true)
 |    |-- DietaryRestrictions: string (nullable = true)
 |    |-- DogsAllowed: string (nullable = true)
 |    |-- DriveThru: string (nullable = true)
 |    |-- GoodForDancing: str

150346

In [16]:
from pyspark.sql import functions as F
#增加引入时间戳和来源，支持数据溯源审计
df_business_bronze = (
    df_business_raw
    .withColumn("_ingest_ts",F.current_timestamp())
    .withColumn("_source_file",F.input_file_name())
    .withColumn("_batch_id",F.lit(batch_id))
)

StatementMeta(, 198cb09d-16bf-47c1-89e6-f5d62861830b, 18, Finished, Available, Finished, False)

In [17]:
df_business_bronze.select("business_id", "_ingest_ts", "_source_file", "_batch_id").show(5, truncate=False)

StatementMeta(, 198cb09d-16bf-47c1-89e6-f5d62861830b, 19, Finished, Available, Finished, False)

+----------------------+--------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------+---------------+
|business_id           |_ingest_ts                |_source_file                                                                                                                                        |_batch_id      |
+----------------------+--------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------+---------------+
|Pns2l4eNsfO8kk83dixA6A|2026-03-12 17:01:25.204265|abfss://2dcf217c-27b0-4bb9-a558-835b9b5dd22c@onelake.dfs.fabric.microsoft.com/e952d822-86a1-4104-a220-8f3d0c6434df/Files/yelp/raw/yelp_business.json|20260312_170110|
|mpf3x-BjTdTEA3yCZrAYPw|2026-03-12 17:01:25.204265|abfss://2dcf217c-27b0-4bb9-a558-835b9b5dd22c@onelake.dfs.fabric.microsoft.com/e95

In [18]:
(
    df_business_bronze.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("business_bronze")
)


StatementMeta(, 198cb09d-16bf-47c1-89e6-f5d62861830b, 20, Finished, Available, Finished, False)

In [20]:
spark.table("business_bronze").printSchema()


StatementMeta(, 198cb09d-16bf-47c1-89e6-f5d62861830b, 22, Finished, Available, Finished, False)

root
 |-- address: string (nullable = true)
 |-- attributes: struct (nullable = true)
 |    |-- AcceptsInsurance: string (nullable = true)
 |    |-- AgesAllowed: string (nullable = true)
 |    |-- Alcohol: string (nullable = true)
 |    |-- Ambience: string (nullable = true)
 |    |-- BYOB: string (nullable = true)
 |    |-- BYOBCorkage: string (nullable = true)
 |    |-- BestNights: string (nullable = true)
 |    |-- BikeParking: string (nullable = true)
 |    |-- BusinessAcceptsBitcoin: string (nullable = true)
 |    |-- BusinessAcceptsCreditCards: string (nullable = true)
 |    |-- BusinessParking: string (nullable = true)
 |    |-- ByAppointmentOnly: string (nullable = true)
 |    |-- Caters: string (nullable = true)
 |    |-- CoatCheck: string (nullable = true)
 |    |-- Corkage: string (nullable = true)
 |    |-- DietaryRestrictions: string (nullable = true)
 |    |-- DogsAllowed: string (nullable = true)
 |    |-- DriveThru: string (nullable = true)
 |    |-- GoodForDancing: str

In [17]:
#Return to 01_0_bronze_run_all
mssparkutils.notebook.exit("SUCCESS")

StatementMeta(, 10e3764a-4d76-4c33-89e0-86c2a7081959, 19, Finished, Available, Finished)

10169368